Get comments from AppStore and PlayStore for the mobile walltes in Bolivia.
* Yolo
* Yape
* Yasta
* Altoke
* TigoM

0. Requirements
1. AppStore scraping
2. PlayStore scraping
3. Merge

In [0]:
%sql
use catalog `workspace`; select * from `default`.`wallet_reviews` limit 100;

## 0. Requirements

In [0]:
%pip install google_play_scraper

In [0]:
dbutils.library.restartPython()

## 1. AppStore scraping

Funtion to download the raw comments from one page

* fetch_reviews(app_id, country, page=1, limit=50)

Function to secuentialy obtain the comments in all pages

* comentarios(app_id, country)

In [0]:
import pandas as pd
import numpy as np
import json, os, uuid
import requests

In [0]:
# 1. Funcion para descargar los comentarios en crudo de una sola pagina
# url = https://itunes.apple.com/bo/rss/customerreviews/page=1/id=1582673945/sortby=mostrecent/json

def fetch_reviews(app_id, country, page=1):
    url = (
        f"https://itunes.apple.com/{country}/rss/customerreviews/"
        f"page={page}/id={app_id}/sortby=mostrecent/json"
    )

    # Define un diccionario llamado headers que contiene un único par clave–valor.
    # La clave "User-Agent" indica al servidor qué tipo de cliente está haciendo la petición.
    # El valor "Mozilla/5.0" simula un navegador; evita que Apple rechace o filtre la solicitud por provenir de un script.
    headers = {"User-Agent": "Mozilla/5.0"}         

    # Envía una petición HTTP GET a la URL especificada en url.
    # El argumento headers=headers incluye el diccionario anterior en la cabecera de la petición.
    # El resultado se almacena en la variable response, que contiene metadatos (código de estado, cabeceras) y el cuerpo de la respuesta.
    response = requests.get(url, headers=headers)   

    # Se exporta la respuesta de la siguiente forma:
    # response.json() convierte el cuerpo JSON de la respuesta en un diccionario de Python.
    # .get("feed", {}) intenta extraer la clave "feed"; si no existe, devuelve un dict vacío para no provocar error.
    # .get("entry", []) extrae la lista de reseñas bajo "entry"; si no está presente, retorna una lista vacía en lugar de None.
    return response.json().get("feed", {}).get("entry", [])

def comentarios(app_id, country):
    # Se inicializan las variables donde se guardara todo y el numero de pagina. Variable page ayuda a iterar cada pagina con comentarios
    all_reviews = []
    page = 1

    # Descargamos todas las páginas de reseñas
    while page<=10:
        batch = fetch_reviews(app_id, country, page=page)
        if not batch:
            break
        all_reviews.extend(batch)   # Es como un append(), pero extend() funciona mejor con Jsons
        page += 1
    # Normalizamos la lista de diccionarios en un DataFrame plano
    return pd.json_normalize(all_reviews)

Rename some columns and filter others.

In [0]:
columns_rename={
    "id.label": "id",
    "author.name.label": "author",
    "updated.label": "date",
    "im:rating.label": "rating",
    "im:version.label": "version",
    "title.label": "title",
    "content.label": "content",
    "im:voteCount.label": "votes_count"
}

columns_filter = [
    "id",
    "author",
    "date",
    "rating",
    "version",
    "title",
    "content",
    "votes_count"
]

In [0]:
# ------------------- Yolo
yolo = comentarios(app_id="1582673945", country="bo")
# Se renombran las columnas
yolo = yolo.rename(columns=columns_rename)
# Se filtran solo las necesarias
yolo = yolo[columns_filter]

# ------------------- Yape
yape = comentarios(app_id="1135987447", country="bo")
yape = yape.rename(columns=columns_rename)
yape = yape[columns_filter]

# ------------------- Yasta
yasta = comentarios(app_id="6502862752", country="bo")
yasta = yasta.rename(columns=columns_rename)
yasta = yasta[columns_filter]

# ------------------- Altoke
altoke = comentarios(app_id="6479173387", country="bo")
altoke = altoke.rename(columns=columns_rename)
altoke = altoke[columns_filter]

# ------------------- TigoM
tigom = comentarios(app_id="1004658616", country="bo")
tigom = tigom.rename(columns=columns_rename)
tigom = tigom[columns_filter]

In [0]:
# Consolidacion de AppStore
# Variable Billetera para identificar cuando se una todo
yolo['Billetera']='Yolo'
yape['Billetera']='Yape'
tigom['Billetera']='TigoM'
yasta['Billetera']='Yasta'
altoke['Billetera']='Altoke'

# Union de todas las tablas
appstore = pd.concat([yolo, yape, tigom, yasta, altoke], ignore_index=True)
#print(wallet_reviews_appstore.shape)
#print(round(wallet_reviews_appstore['Billetera'].value_counts(normalize=True) * 100,1))

## 2. PlayStore scraping

In [0]:
import pandas as pd
from google_play_scraper import Sort, reviews_all, reviews, app
#from app_store_scraper import AppStore
import numpy as np
import json, os, uuid

In [0]:
# ------------------- Yolo
yolo = reviews_all(
    'bo.com.yolopago',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, # defaults to 0
    sort=Sort.MOST_RELEVANT # defaults to Sort.MOST_RELEVANT
    #filter_score_with=5 # defaults to None(means all score)
)

# ------------------- Yape
yape = reviews_all(
    'com.bcp.bo.wallet',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 
   
)

# ------------------- Yasta
yasta = reviews_all(
    'com.busa.wallet',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 
)

# ------------------- Altoke
altoke= reviews_all(
    'com.bancosol.altoke',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 

)

# ------------------- TigoM
tigom= reviews_all(
    'bo.tigo.mfsapp',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 
)


In [0]:
# Convert to tables
yolo_tabla=pd.json_normalize(yolo)
yolo_tabla['Billetera']='Yolo'

yape_tabla=pd.json_normalize(yape)
yape_tabla['Billetera']='Yape'

tigom_tabla=pd.json_normalize(tigom)
tigom_tabla['Billetera']='TigoM'

yasta_tabla=pd.json_normalize(yasta)
yasta_tabla['Billetera']='Yasta'

altoke_tabla=pd.json_normalize(altoke)
altoke_tabla['Billetera']='Altoke'

In [0]:
# Concatenacion
playstore = pd.concat([yolo_tabla, yape_tabla, tigom_tabla, yasta_tabla, altoke_tabla], ignore_index=True)

## 3. Merge

In [0]:
# Filter columns
appstore=appstore[['id','date','rating','version','content','Billetera']]
appstore['Store']='AppStore'

playstore=playstore[['reviewId','at','score','reviewCreatedVersion','content','Billetera']]
playstore['Store']='PlayStore'

# Rename columns
playstore = playstore.rename(
    columns={
        'reviewId': 'id',
        'at': 'date',
        'score': 'rating',
        'reviewCreatedVersion': 'version'
    }
)

# Concat by row
Stores = pd.concat([appstore, playstore], ignore_index=True)

Stores.head()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, FloatType

Stores_clean = Stores.copy()

schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("rating", FloatType(), True),
    StructField("version", StringType(), True),
    StructField("content", StringType(), True),
    StructField("Billetera", StringType(), True),
    StructField("Store", StringType(), True)
])

#Stores_clean['date'] = pd.to_datetime(Stores_clean['date'], errors='coerce')
#Stores_clean['rating'] = pd.to_numeric(Stores_clean['rating'], errors='coerce')

df = spark.createDataFrame(Stores_clean, schema=schema)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.wallet_reviews")